In [ ]:
pip install ollama

In [ ]:
!pip install PyPDF2
!pip install langchain langchain-openai

In [ ]:
!pip install -U \
  langchain \
  langchain-openai \
  langchain-community \
  langgraph \
  faiss-cpu \
  pypdf \
  python-dotenv

In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
pip install sentence-transformers faiss-cpu

In [ ]:
import os
import re
import json
from typing import List, TypedDict, Literal

from pydantic import BaseModel, Field

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, START, END
from ollama import Client

from dotenv import load_dotenv
load_dotenv()

In [ ]:
from ollama import Client

class OllamaLLM:
    def __init__(self, model="qwen3.5"):
        self.client = Client(host="https://api.ollama.com")
        self.model = model

    def invoke(self, prompt):
        response = self.client.chat(
            model=self.model,
            messages=[
                {"role": "user", "content": prompt}
            ],
        )
        return response["message"]["content"]

In [ ]:
llm = OllamaLLM()

In [ ]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

if file_name.endswith(".pdf"):
    loader = PyPDFLoader(file_name)
    docs = loader.load()
    print(f"Loaded {len(docs)} pages")
else:
    print("Please upload a PDF file")

In [ ]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

In [ ]:
class State(TypedDict):
    question: str
    need_retrieval: bool

    docs: List[Document]
    relevant_docs: List[Document]

    context: str
    answer: str

    web_query: str

In [ ]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external data is required, else False."
    )

def build_decide_prompt(question: str) -> str:
    return f"""
You are a decision system in a RAG pipeline.

Decide whether answering the question requires external retrieval.

OUTPUT:
Return ONLY JSON:
{{"should_retrieve": true}} or {{"should_retrieve": false}}

RULES:

Return TRUE only if:
- Question depends on documents, PDFs, files
- Requires exact facts, citations, or up-to-date info
- Mentions "this document", "above", "file", etc.

Return FALSE if:
- General knowledge (e.g., "What is recursion?")
- Conceptual explanation
- Can be answered without external context

IMPORTANT:
- Default to FALSE
- Do NOT explain
- Do NOT hallucinate

Question:
{question}
"""

def decide_retrieval(state: State):
    prompt = build_decide_prompt(state["question"])
    response = llm.invoke(prompt)

    try:
        match = re.search(r"\{.*\}", response, re.DOTALL)
        if match:
            data = json.loads(match.group())
            decision = RetrieveDecision(**data)
            return {"need_retrieval": decision.should_retrieve}
    except:
        pass

    return {"need_retrieval": False}

In [ ]:
direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a reliable AI assistant.\n\n"

            "Answer using ONLY general knowledge.\n"
            "Do NOT assume external documents.\n\n"

            "RULES:\n"
            "- Answer only if confident\n"
            "- Keep answer concise (1–2 sentences)\n"
            "- No guessing\n\n"

            "FAILURE RESPONSE:\n"
            "I don't know based on my general knowledge."
        ),
        ("human", "{question}"),
    ]
)

def generate_direct(state: State):
    messages = direct_generation_prompt.format_messages(
        question=state["question"]
    )

    prompt = "\n".join([m.content for m in messages])
    out = llm.invoke(prompt)

    return {
        **state,
        "answer": out
    }

In [ ]:
def retrieve(state: State):
    return {
        **state,
        "docs": retriever.invoke(state["question"])
    }

In [ ]:
class RelevanceDecision(BaseModel):
    is_relevant: bool

def build_relevance_prompt(question: str, document: str) -> str:
    return f"""
You are a strict relevance classifier.

Decide if the document helps answer the question.

OUTPUT:
Return ONLY JSON:
{{"is_relevant": true}} or {{"is_relevant": false}}

RULES:
- TRUE if document contains useful or related info
- FALSE if unrelated or weakly related
- Prefer FALSE if unsure

Question:
{question}

Document:
{document}
"""

def is_relevant(state: State):
    relevant_docs = []

    for doc in state["docs"]:
        prompt = build_relevance_prompt(
            state["question"],
            doc.page_content
        )

        response = llm.invoke(prompt)

        try:
            match = re.search(r"\{.*\}", response, re.DOTALL)
            if match:
                data = json.loads(match.group())
                decision = RelevanceDecision(**data)

                if decision.is_relevant:
                    relevant_docs.append(doc)
        except:
            pass

    return {
        **state,
        "relevant_docs": relevant_docs
    }

In [ ]:
rag_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a RAG assistant.\n\n"

            "Answer using ONLY the provided context.\n\n"

            "RULES:\n"
            "- Use available information even if partial\n"
            "- Combine multiple pieces if needed\n"
            "- Keep answer concise (1–2 sentences)\n"
            "- Do NOT use outside knowledge\n\n"

            "FAIL ONLY IF:\n"
            "- Context is completely unrelated\n\n"

            "FAIL RESPONSE:\n"
            "No relevant document found."
        ),
        (
            "human",
            "Question:\n{question}\n\nContext:\n{context}"
        ),
    ]
)

def generate_from_context(state: State):
    context = "\n\n---\n\n".join(
        [d.page_content for d in state.get("relevant_docs", [])]
    ).strip()

    if not context:
        return {
            **state,
            "answer": "No relevant document found.",
            "context": ""
        }

    messages = rag_generation_prompt.format_messages(
        question=state["question"],
        context=context
    )

    prompt = "\n".join([m.content for m in messages])
    out = llm.invoke(prompt)

    return {
        **state,
        "answer": out,
        "context": context
    }

In [ ]:
class WebQuery(BaseModel):
    query: str


def build_rewrite_prompt(question: str) -> str:
    return f"""
You are a search query generator.

Your task is to convert a user question into a concise web search query.

OUTPUT FORMAT:
Return ONLY valid JSON:
{{"query": "your search query"}}

STRICT RULES:
- Output MUST be valid JSON (no explanation, no extra text)
- Query must be 6 to 14 words
- Use keyword-style phrasing (not full sentences)
- Preserve important terms (names, topics, concepts)
- If the question implies recent information, append: (last 30 days)
- DO NOT answer the question
- DO NOT explain anything

GOOD EXAMPLES:
Question: What is recursion in programming?
{{"query": "recursion programming definition explanation base case example"}}

Question: latest AI news
{{"query": "latest AI news developments trends 2025 (last 30 days)"}}

BAD OUTPUT:
- Any explanation
- Missing JSON
- Multiple keys

Question:
{question}
"""


def rewrite_query_node(state: State):
    prompt = build_rewrite_prompt(state["question"])

    response = llm.invoke(prompt)

    try:
        data = json.loads(response)
        query = data.get("query", "").strip()

        if not query:
            raise ValueError("Empty query")

        return {
            **state,
            "web_query": query
        }

    except:
        # fallback: use original question
        return {
            **state,
            "web_query": state["question"]
        }


tavily = TavilySearchResults(max_results=5)

def web_search_node(state: State):
    q = state.get("web_query") or state["question"]

    results = tavily.invoke({"query": q})

    docs = []
    for r in results or []:
        title = r.get("title", "")
        url = r.get("url", "")
        content = r.get("content", "") or r.get("snippet", "")

        text = f"TITLE: {title}\nURL: {url}\nCONTENT:\n{content}"

        docs.append(
            Document(
                page_content=text,
                metadata={"source": "web", "url": url, "title": title},
            )
        )

    return {"docs": docs}

In [ ]:
def no_relevant_docs(state: State):
    return {"answer": "No relevant document found.", "context": ""}

In [ ]:
def route_after_decide(state: State):
    if state["need_retrieval"]:
        return "retrieve"
    return "generate_direct"

In [ ]:
def route_after_relevance(state: State) -> Literal["generate_from_context", "rewrite_query"]:
    if state.get("relevant_docs") and len(state["relevant_docs"]) > 0:
        return "generate_from_context"
    return "rewrite_query"

In [ ]:
g = StateGraph(State)

g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)

g.add_node("is_relevant", is_relevant)
g.add_node("generate_from_context", generate_from_context)

g.add_node("rewrite_query", rewrite_query_node)
g.add_node("web_search", web_search_node)

g.add_edge(START, "decide_retrieval")

g.add_conditional_edges(
    "decide_retrieval",
    route_after_decide,
    {
        "generate_direct": "generate_direct",
        "retrieve": "retrieve",
    },
)

g.add_edge("generate_direct", END)

g.add_edge("retrieve", "is_relevant")

g.add_conditional_edges(
    "is_relevant",
    route_after_relevance,
    {
        "generate_from_context": "generate_from_context",
        "rewrite_query": "rewrite_query",
    },
)

# web fallback path
g.add_edge("rewrite_query", "web_search")
g.add_edge("web_search", "is_relevant")

g.add_edge("generate_from_context", END)

app = g.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": "Answer in one sentence only: according to the document, what is the role of the handle_client function?",
        "docs": [],
        "relevant_docs": [],
        "context": "",
        "answer": "",
    }
)

print(result["answer"])

In [ ]:
for doc in result['relevant_docs']:
    print(doc.page_content)
    print("*"*100)

In [ ]:
len(result['relevant_docs'])

In [ ]:
result = app.invoke(
    {
        "question": "Who won the Aus vs Zim World T20 match 2026 and who was the top scorer",
        "docs": [],
        "relevant_docs": [],
        "context": "",
        "answer": "",
    }
)

print(result["answer"])

In [ ]:
print(result["web_query"])
print(len(result["docs"]))
print(len(result["relevant_docs"]))